<a href="https://colab.research.google.com/github/0xShug0/audio.cpp/blob/main/Notebooks/colab_audio_cpp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# audio.cpp WebUI
Run the cell below to build audio.cpp with CUDA and open its WebUI. The public link remains active while the cell is running. Anyone with the link can use the WebUI, so do not share it.

In [ ]:
# @title Start audio.cpp

import json
import os
import platform
import re
import hashlib
import shutil
import socket
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path

from IPython.display import HTML, clear_output, display

REPO_DIR = Path("/content/audio.cpp")
REPO_URL = "https://github.com/0xShug0/audio.cpp.git"
REPO_BRANCH = "main"
BUILD_DIR = REPO_DIR / "build/colab-cuda"
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_socket:
    port_socket.bind(("127.0.0.1", 0))
    SERVER_PORT = port_socket.getsockname()[1]
BUILD_LOG = Path("/content/audio_cpp_build.log")
SERVER_LOG = Path("/content/audio_cpp_server.log")
TUNNEL_LOG = Path("/content/audio_cpp_tunnel.log")
CLOUDFLARED = Path("/content/cloudflared")
RELEASE_API_URL = "https://api.github.com/repos/0xShug0/audio.cpp/releases/latest"
PREBUILT_ASSET_SUFFIX = "-bin-ubuntu-x64-cuda12.8-colab.tar.gz"
PREBUILT_CUDA_ARCHS = {"75"}
PREBUILT_ROOT = Path("/content/audio-cpp-prebuilt")

def latest_colab_prebuilt():
    request = urllib.request.Request(
        RELEASE_API_URL,
        headers={"Accept": "application/vnd.github+json", "User-Agent": "audio.cpp-colab"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        release = json.load(response)
    matches = [asset for asset in release.get("assets", []) if asset["name"].endswith(PREBUILT_ASSET_SUFFIX)]
    if len(matches) != 1:
        raise RuntimeError("The latest release does not contain a Colab CUDA prebuilt.")
    asset = matches[0]
    digest = asset.get("digest", "")
    if not digest.startswith("sha256:"):
        raise RuntimeError("The Colab CUDA prebuilt does not have a SHA-256 digest.")
    return asset["browser_download_url"], digest.removeprefix("sha256:")

def run(command, *, cwd=None, env=None, log=None):
    if log is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    with log.open("a", encoding="utf-8") as output:
        try:
            subprocess.run(
                command, cwd=cwd, env=env, check=True, stdout=output, stderr=subprocess.STDOUT
            )
        except subprocess.CalledProcessError as error:
            command_text = " ".join(map(str, command))
            raise RuntimeError(f"Command failed: {command_text}\n\n{tail(log)}") from error

def tail(path, line_count=40):
    if not path.exists():
        return ""
    return "".join(path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-line_count:])

def run_build(command, *, cwd, env, log):
    progress_pattern = re.compile(r"\[(\d+)/(\d+)\]")
    with log.open("a", encoding="utf-8") as output:
        process = subprocess.Popen(
            command, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT
        )
        last_progress = None
        while process.poll() is None:
            matches = progress_pattern.findall(
                log.read_text(encoding="utf-8", errors="replace")
            )
            if matches and matches[-1] != last_progress:
                completed, total = map(int, matches[-1])
                width = 30
                filled = min(width, completed * width // total)
                bar = "#" * filled + "-" * (width - filled)
                percent = completed * 100 // total
                clear_output(wait=True)
                print(f"Building audio.cpp with CUDA [{bar}] {percent}% ({completed}/{total})")
                last_progress = matches[-1]
            time.sleep(1)
        return_code = process.wait()
    if return_code != 0:
        command_text = " ".join(map(str, command))
        raise RuntimeError(f"Command failed: {command_text}\n\n{tail(log)}")

def stop(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

if subprocess.run(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU was detected. In Colab, select Runtime > Change runtime type > T4 GPU, "
        "restart the session, and run this cell again."
    )

if platform.machine() != "x86_64":
    raise RuntimeError(f"This notebook currently supports x86_64 Colab runtimes, not {platform.machine()}.")

BUILD_LOG.write_text("", encoding="utf-8")
SERVER_LOG.write_text("", encoding="utf-8")
TUNNEL_LOG.write_text("", encoding="utf-8")

print("Preparing the Colab runtime...", flush=True)
run(["apt-get", "update", "-qq"], log=BUILD_LOG)
run(
    [
        "apt-get", "install", "-y", "-qq",
        "ca-certificates", "curl", "ffmpeg", "libgomp1", "libssl3",
    ],
    log=BUILD_LOG,
)

compute_capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).splitlines()[0].strip()
cuda_arch = compute_capability.replace(".", "")
if not cuda_arch.isdigit():
    raise RuntimeError(f"Could not determine the CUDA architecture from: {compute_capability!r}")

server_binary = None
if cuda_arch in PREBUILT_CUDA_ARCHS:
    archive_path = None
    try:
        prebuilt_url, prebuilt_sha256 = latest_colab_prebuilt()
        archive_path = Path("/content") / prebuilt_url.rsplit("/", 1)[-1]
        print("Downloading the CUDA prebuilt...", flush=True)
        run(["curl", "-L", "--fail", "--silent", "--show-error", prebuilt_url, "-o", str(archive_path)], log=BUILD_LOG)
        actual_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
        if actual_sha256 != prebuilt_sha256:
            raise RuntimeError("The downloaded CUDA prebuilt failed SHA-256 verification.")
        if PREBUILT_ROOT.exists():
            shutil.rmtree(PREBUILT_ROOT)
        PREBUILT_ROOT.mkdir(parents=True)
        with tarfile.open(archive_path, "r:gz") as archive:
            destination = PREBUILT_ROOT.resolve()
            for member in archive.getmembers():
                member_path = (PREBUILT_ROOT / member.name).resolve()
                if destination not in member_path.parents and member_path != destination:
                    raise RuntimeError("The CUDA prebuilt contains an unsafe path.")
            archive.extractall(PREBUILT_ROOT, filter="data")
        matches = list(PREBUILT_ROOT.rglob("audiocpp_server"))
        if len(matches) != 1:
            raise RuntimeError("The CUDA prebuilt does not contain one audiocpp_server binary.")
        server_binary = matches[0]
        server_binary.chmod(0o755)
    except Exception as error:
        print(f"The CUDA prebuilt could not be used; building from source instead. ({error})", flush=True)
        server_binary = None
    finally:
        if archive_path is not None:
            archive_path.unlink(missing_ok=True)

if server_binary is None:
    run(
        [
            "apt-get", "install", "-y", "-qq", "build-essential", "cmake",
            "libssl-dev", "ninja-build", "software-properties-common",
        ],
        log=BUILD_LOG,
    )
    if subprocess.run(["bash", "-lc", "command -v gcc-13 && command -v g++-13"], stdout=subprocess.DEVNULL).returncode != 0:
        run(["add-apt-repository", "-y", "ppa:ubuntu-toolchain-r/test"], log=BUILD_LOG)
        run(["apt-get", "update", "-qq"], log=BUILD_LOG)
    run(["apt-get", "install", "-y", "-qq", "gcc-13", "g++-13"], log=BUILD_LOG)

    if not (REPO_DIR / ".git").exists():
        if REPO_DIR.exists():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Restart the runtime and try again.")
        print("Downloading audio.cpp source...", flush=True)
        run(["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], log=BUILD_LOG)
    else:
        print("Using the existing audio.cpp checkout.", flush=True)

    build_env = os.environ.copy()
    build_env.update({"CC": "gcc-13", "CXX": "g++-13", "CUDAHOSTCXX": "g++-13"})
    jobs = max(1, min(os.cpu_count() or 2, 4))
    server_binary = BUILD_DIR / "bin/audiocpp_server"
    print("Building audio.cpp with CUDA. This can take several minutes...", flush=True)
    run_build(
        [
            "bash", "scripts/build_linux.sh",
            "--backend", "cuda",
            "--build-dir", str(BUILD_DIR),
            "--build-type", "Release",
            "--cuda-arch", cuda_arch,
            "--deployment-build",
            "--native-model-manager",
            "--system-openssl",
            "--target", "audiocpp_server",
            "--jobs", str(jobs),
        ],
        cwd=REPO_DIR,
        env=build_env,
        log=BUILD_LOG,
    )
if not CLOUDFLARED.exists():
    print("Installing the public tunnel...", flush=True)
    run(
        [
            "curl", "-L", "--fail", "--silent", "--show-error",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", str(CLOUDFLARED),
        ],
        log=BUILD_LOG,
    )
    CLOUDFLARED.chmod(0o755)

server_process = None
tunnel_process = None
server_output = None
tunnel_output = None

try:
    print("Starting the audio.cpp WebUI...", flush=True)
    server_output = SERVER_LOG.open("w", encoding="utf-8")
    server_process = subprocess.Popen(
        [
            str(server_binary), "--ui", "--ui-management",
            "--backend", "cuda", "--host", "127.0.0.1",
            "--port", str(SERVER_PORT),
        ],
        cwd=server_binary.parent,
        stdout=server_output,
        stderr=subprocess.STDOUT,
        text=True,
    )

    health_url = f"http://127.0.0.1:{SERVER_PORT}/health"
    deadline = time.monotonic() + 120
    while time.monotonic() < deadline:
        if server_process.poll() is not None:
            raise RuntimeError("audio.cpp stopped during startup.\n\n" + tail(SERVER_LOG))
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("audio.cpp did not become ready.\n\n" + tail(SERVER_LOG))

    tunnel_output = TUNNEL_LOG.open("w", encoding="utf-8")
    tunnel_process = subprocess.Popen(
        [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--no-autoupdate"],
        stdout=tunnel_output,
        stderr=subprocess.STDOUT,
        text=True,
    )

    public_url = None
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("The public tunnel stopped during startup.\n\n" + tail(TUNNEL_LOG))
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text(encoding="utf-8", errors="replace"))
        if match:
            public_url = match.group(0)
            break
        time.sleep(1)
    if public_url is None:
        raise RuntimeError("The public tunnel did not provide a URL.\n\n" + tail(TUNNEL_LOG))

    clear_output(wait=True)
    display(HTML(
        f'<h2>audio.cpp is running</h2>'
        f'<p><a href="{public_url}" target="_blank" rel="noopener noreferrer">Open the audio.cpp WebUI</a></p>'
        '<p>Keep this cell running while you use the WebUI. Anyone with the link can access this session.</p>'
    ))

    while True:
        if server_process.poll() is not None:
            raise RuntimeError("audio.cpp stopped unexpectedly.\n\n" + tail(SERVER_LOG))
        if tunnel_process.poll() is not None:
            raise RuntimeError("The public tunnel stopped unexpectedly.\n\n" + tail(TUNNEL_LOG))
        time.sleep(5)
except KeyboardInterrupt:
    clear_output(wait=True)
    print("audio.cpp stopped.")
finally:
    stop(tunnel_process)
    stop(server_process)
    if tunnel_output is not None:
        tunnel_output.close()
    if server_output is not None:
        server_output.close()
